In [1]:
from collections import defaultdict

class KneserNeyLanguageModel:
    def __init__(self, n=5, discount=0.75):
        self.n = n
        self.discount = discount
        self.ngram_counts = [defaultdict(int) for _ in range(n)]
        self.context_counts = [defaultdict(int) for _ in range(n)]
        self.continuation = defaultdict(set)
        self.vocab = set()

    def tokenize(self, text):
        return text.lower().split()

    def train(self, corpus):
        for sentence in corpus:
            tokens = ["<s>"] * (self.n - 1)
            tokens += self.tokenize(sentence)
            tokens += ["</s>"]

            for token in tokens:
                self.vocab.add(token)

            length = len(tokens)

            for i in range(length):
                for k in range(1, self.n + 1):
                    if i + k <= length:
                        gram = tuple(tokens[i:i + k])
                        self.ngram_counts[k - 1][gram] += 1

                        if k > 1:
                            context = gram[:-1]
                            self.context_counts[k - 1][context] += 1
                            self.continuation[gram[-1]].add(context)

    def continuation_probability(self, word):
        total = sum(len(v) for v in self.continuation.values())

        if total == 0:
            return 1 / max(len(self.vocab), 1)

        return len(self.continuation[word]) / total

    def kn_probability(self, history, word):
        history = tuple(history)
        order = len(history) + 1

        if order == 1:
            return self.continuation_probability(word)

        gram = history + (word,)
        gram_count = self.ngram_counts[order - 1].get(gram, 0)
        context_count = self.context_counts[order - 1].get(history, 0)

        if context_count == 0:
            return self.kn_probability(history[1:], word)

        unique_followers = len(
            [g for g in self.ngram_counts[order - 1] if g[:-1] == history]
        )

        lambda_weight = self.discount * unique_followers / context_count
        discounted = max(gram_count - self.discount, 0) / context_count
        lower = self.kn_probability(history[1:], word)

        return discounted + lambda_weight * lower

    def predict(self, text, top_k=5):
        tokens = self.tokenize(text)

        if len(tokens) >= self.n - 1:
            history = tokens[-(self.n - 1):]
        else:
            history = ["<s>"] * ((self.n - 1) - len(tokens))
            history += tokens

        scores = []

        for word in self.vocab:
            if word == "<s>":
                continue

            prob = self.kn_probability(history, word)
            scores.append((word, prob))

        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

    def add_new_word(self, word):
        self.vocab.add(word)


queries = [
    "best places to visit",
    "best places to visit in",
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit during summer",
    "best places to visit near me",
    "best restaurants near me",
    "best restaurants in chennai",
    "best hotels in india",
    "best hotels near airport"
]

model = KneserNeyLanguageModel()
model.train(queries)

model.add_new_word("iphone17")
model.add_new_word("chatgpt6")

query = "best places to visit"

print("Input Query:", query)
print("Suggestions:")

for word, score in model.predict(query):
    print(word, round(score, 6))

Input Query: best places to visit
Suggestions:
in 0.709481
</s> 0.096133
near 0.084481
during 0.080244
me 0.003708
